# DeltaNet v2: Performance Benchmark on Google Colab (Tesla T4)
### Comparing DeltaNet v2 (SM75 CuTe) against NVIDIA GatedDeltaNet-2 and Flash Linear Attention

> **Environment & Build Strategy**: This notebook targets **C++20** with the latest PyTorch runtime by default. If the compiler or environment encounters compatibility issues, it automatically falls back to **C++17** and PyTorch 2.4.1.

This notebook downloads and executes the benchmark suite from `/content/Deltanet/tests/` to evaluate:
- **Runtime Performance**: Latency (ms) and Throughput (tokens/sec)
- **Peak Memory Consumption**: VRAM allocated & reserved (MB)

**Baselines Compared:**
1. **DeltaNet v2 (Ours)**: Fused CUDA kernels targeting NVIDIA Turing SM75 (Tesla T4, RTX 20-series) with independent $k, v$ gating.
2. **FLA GatedDeltaNet-2**: NVIDIA's official Gated DeltaNet-2 paper architecture implemented via Flash Linear Attention (`fla.layers.GatedDeltaNet2`).
3. **FLA GatedDeltaNet (v1)**: Standard Gated DeltaNet linear attention layer (`fla.layers.GatedDeltaNet`).
4. **PyTorch Reference**: Unfused PyTorch eager baseline.

## 1. Verify GPU Hardware
Confirm that Google Colab is running with an **NVIDIA Tesla T4** GPU (SM75, 16 GB VRAM).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Build DeltaNet v2
Clones `https://github.com/CBG6682/Varlen-Gated-DeltaNet.git` and compiles the SM75 fused CUDA extensions.

**Compilation Strategy**:
- **Primary**: Attempts compilation with **C++20** on the active PyTorch environment.
- **Fallback**: If C++20 compilation fails, it automatically installs PyTorch 2.4.1 and rebuilds with **C++17**.
- **Resource Protection**: Enforces `MAX_JOBS=2` to prevent Colab CPU and RAM exhaustion.

In [ ]:
%%bash
# 1. Clone repository with dedicated third_party/cutlass submodule
rm -rf /content/Deltanet
git clone --recursive https://github.com/CBG6682/Varlen-Gated-DeltaNet.git /content/Deltanet
cd /content/Deltanet

# 2. Ensure third_party/cutlass submodule is initialized
git submodule update --init --depth 1 third_party/cutlass || true

# 3. Install build dependencies
pip install -q ninja setuptools wheel einops

# 4. Compile with C++20; automatically fallback to C++17 on failure
export MAX_JOBS=2
export CXX_STANDARD=c++20
echo "===> [1/2] Attempting compilation with C++20..."

if pip install -v . --no-build-isolation; then
    echo "===> [SUCCESS] Built DeltaNet v2 with C++20!"
else
    echo "===> [FALLBACK] C++20 compilation failed. Falling back to C++17 with PyTorch 2.4.1..."
    pip install -q torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    export CXX_STANDARD=c++17
    export FORCE_REBUILD=1
    pip install -v . --no-build-isolation
    echo "===> [SUCCESS] Built DeltaNet v2 with C++17!"
fi

# Copy compiled .so into local package directory for in-tree imports
cp build/lib.*/deltanet_v2/_C*.so deltanet_v2/ 2>/dev/null || true


## 3. (Optional) Package Standalone Wheel (.whl)
Packages the compiled extension into a clean `.whl` distribution strictly restricted to `deltanet_v2/` (excluding foreign container files) and downloads it.

In [ ]:
# ==============================================================================
# Package clean Wheel (.whl) containing only deltanet_v2
# ==============================================================================
import os
import glob
import zipfile
from google.colab import files

%cd /content/Deltanet

# Clean previous build directories
!rm -rf dist build *.egg-info

# Build wheel package (setup.py strictly restricts packages to deltanet_v2)
!python setup.py bdist_wheel

# Inspect wheel contents to verify no outside container files were included
wheel_files = glob.glob("dist/*.whl")
if wheel_files:
    target_whl = wheel_files[0]
    print("\n" + "=" * 60)
    print(f"VERIFYING WHEEL CONTENTS: {os.path.basename(target_whl)}")
    print("=" * 60)
    with zipfile.ZipFile(target_whl, "r") as z:
        for member in z.namelist():
            print(f"  - {member}")
    print("=" * 60)

    # Trigger download to local machine
    print(f"\n>> Downloading wheel: {os.path.basename(target_whl)}...")
    files.download(target_whl)
else:
    print("[!] No wheel file found in /content/Deltanet/dist/!")

## 4. Install Baseline Libraries
Installs **Flash Linear Attention** (`fla`) and **NVIDIA GatedDeltaNet-2** (`NVlabs/GatedDeltaNet-2`).

In [ ]:
# Install Flash Linear Attention (FLA)
!pip install -q flash-linear-attention

# Install NVlabs GatedDeltaNet-2
!pip install -q git+https://github.com/nvlabs/gateddeltanet-2.git --no-deps || true

## 5. Numerical Correctness Sanity Test
Verifies numerical consistency between DeltaNet v2 CUDA kernels and the pure PyTorch reference.

In [ ]:
!python /content/Deltanet/tests/test_correctness.py

## 6. Benchmark: Autoregressive Inference (L = 1)
Measures single-step decode latency (ms/step), generation throughput (tokens/sec), and peak VRAM across batch sizes $B \in [1, 16, 64, 256, 1024, 2048]$.

In [ ]:
!python /content/Deltanet/tests/bench_inference.py --dim 768 --q-heads 8 --kv-heads 8 --batches 1,16,64,256,1024,2048

## 7. Benchmark: Training Pass (Forward + Backward)
Measures step time (ms), training throughput (tokens/sec), and peak VRAM across sequence lengths $L \in [512, 1024, 2048, 4096]$.

In [ ]:
!python /content/Deltanet/tests/bench_training.py --dim 768 --q-heads 8 --kv-heads 8 --batch-size 4 --seq-lens 512,1024,2048,4096

## 8. Run Full Benchmark Suite
Runs all benchmarks and prints consolidated comparison tables.

In [ ]:
!python /content/Deltanet/tests/bench_all.py